# CS781 Phase 4 — Experiment 3 (Reviewed v3)
## Model2 + AdaLoRA + Hierarchical Head/Middle/Evidence/Tail CJPE

This notebook keeps the successful Phase-4 long-document pipeline and changes the backbone/adaptation method to the exact Phase-1 **Model2 decoder-only causal Transformer + AdaLoRA**.

**Model2:** vocab 24,576; d_model 640; 13 layers; 10 heads; FFN 2560; sinusoidal positions; LayerNorm; GELU; fused `qkv`; max positions 2048; ~79.7M parameters.

**AdaLoRA:** each selected frozen linear gets `ΔW = B diag(E) A`, initial rank 12. A global adaptive rank allocator uses EMA importance `|E·grad(E)|` and prunes rank directions during training toward average target rank 4.

Targets in every block: `attn.qkv`, `attn.out_proj`, `ff[0]`, `ff[2]`.

Long-document representation: 4 × 384-token views — **HEAD / MIDDLE / EVIDENCE / TAIL** — followed by learned chunk attention + max pooling.

Leakage-safe split is retained: 32,305 deduplicated TRAIN, 994 clean DEV, 1,517 TEST. TEST labels/expert fields and DEV labels for TEST-overlap IDs are never used.

At most 3 high-value submissions are produced:
- `submission_best.zip` — calibrated heldout+refit margin ensemble.
- `submission_heldout.zip` — clean-DEV-selected heldout model.
- `submission_margin_h70_r30.zip` — 70/30 calibrated heldout/refit margin hedge.


## v3 reviewed update — core experiment unchanged

Core objective is unchanged:
**Phase-1 Model2 + adaptive-rank AdaLoRA + Head/Middle/Evidence/Tail hierarchical CJPE.**

Useful v2 improvements retained:
- safer JSON helpers;
- tokenizer ambiguity warning + optional SHA pin;
- Phase-1 LM sanity check before expensive training;
- non-finite micro-batch handling without breaking accumulation/eval boundaries;
- finite-only AdaLoRA importance updates;
- string IDs in NPZ;
- richer manifest.

Additional v3 improvements:
- LM sanity now hard-stops only for a clearly broken tokenizer/model pairing;
- rank-budget invariants are asserted after pruning;
- final refit checkpoint metadata is richer and remains atomic;
- third submission is a **70% heldout / 30% refit calibrated-margin hedge** instead of raw probability averaging;
- ZIP contents are re-opened and structurally verified;
- manifest records all submission strategies and disagreements.

No change to AdaLoRA rank 12→4, alpha 24, target modules, 13-layer Model2,
4×384 chunking, leakage-safe split, optimizer, threshold search, or refit-duration logic.


In [1]:
# CELL 1 — imports/config
import os,re,gc,json,math,time,random,zipfile,subprocess,sys,hashlib,tempfile
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from sklearn.metrics import f1_score,precision_score,recall_score
from huggingface_hub import hf_hub_download,login

SEED=1234
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

def save_json(obj,path,**kw):
    with open(path,"w",encoding="utf-8") as f: json.dump(obj,f,**kw)
def read_json(path):
    with open(path,encoding="utf-8") as f: return json.load(f)

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",DEVICE)
if DEVICE.type=="cuda":
    print("GPU:",torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32=True
    torch.backends.cudnn.allow_tf32=True
torch.set_float32_matmul_precision("high")

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN=None
assert HF_TOKEN,"HF_TOKEN missing in Kaggle Secrets"
login(token=HF_TOKEN)

HF_REPO="Exploration-Lab/CS-781-Capstone"
ROOT=Path("/kaggle/working/phase4_exp3_model2_adalora")
ROOT.mkdir(parents=True,exist_ok=True)

MODEL2_CKPT_OVERRIDE=None
TOKENIZER_OVERRIDE="/kaggle/input/datasets/aaryant23/phase2-model2-ckpts/tokenizer (1).json"
EXPECTED_TOKENIZER_SHA256=None   

VOCAB_SIZE=24576
D_MODEL=640
N_LAYERS=13
N_HEADS=10
FFN_DIM=2560
DROPOUT=0.0
MAX_POSITIONS=2048

N_CHUNKS=4
CHUNK_LEN=384
CHUNK_NAMES=["head","middle","evidence","tail"]
CASE_MODE="auto"
COLLAPSE_WS=True
EVIDENCE_CUES=[
    "appeal is allowed","appeal allowed",
    "appeal is dismissed","appeal dismissed",
    "petition is allowed","petition allowed",
    "petition is dismissed","petition dismissed",
    "we allow","we dismiss",
    "is hereby allowed","is hereby dismissed",
    "set aside","acquitted","convicted",
    "in the result","accordingly",
]

ADALORA_INIT_R=12
ADALORA_TARGET_R=4
ADALORA_ALPHA=24.0
ADALORA_DROPOUT=0.05
ADALORA_LR=2.0e-4
ADALORA_TARGETS=("attn.qkv","attn.out_proj","ff.0","ff.2")
RANK_WARMUP_FRAC=0.10
RANK_FINAL_FRAC=0.20
RANK_UPDATE_INTERVAL=10
IMPORTANCE_BETA=0.90

CHUNK_PROJ_DIM=384
HEAD_DROPOUT=0.15
HEAD_LR=5.0e-4

NUM_EPOCHS=2
EVAL_PER_EPOCH=3
PATIENCE_EVALS=4
MICRO_BATCH_SIZE=2
GRAD_ACCUM_STEPS=8
EVAL_BATCH_SIZE=6
WARMUP_RATIO=0.06
WEIGHT_DECAY=0.01
GRAD_CLIP=1.0
LABEL_SMOOTHING=0.02
USE_AMP=True
AMP_INIT_SCALE=4096.0
NUM_WORKERS=2

THRESHOLDS=np.round(np.arange(0.20,0.801,0.005),3)
THRESH_SMOOTH=9
RUN_REFIT=True
REFIT_MATCH_ORIGINAL_SCHEDULE=True

print("chunks:",N_CHUNKS,"x",CHUNK_LEN)
print("effective document batch:",MICRO_BATCH_SIZE*GRAD_ACCUM_STEPS)
print("AdaLoRA init→target rank:",ADALORA_INIT_R,"→",ADALORA_TARGET_R)
print("AdaLoRA targets:",ADALORA_TARGETS)


Device: cuda
GPU: Tesla T4
chunks: 4 x 384
effective document batch: 16
AdaLoRA init→target rank: 12 → 4
AdaLoRA targets: ('attn.qkv', 'attn.out_proj', 'ff.0', 'ff.2')


In [2]:
# CELL 2 — locate Model2 checkpoint/tokenizer
def all_files():
    for base in ["/kaggle/input","/kaggle/working"]:
        if not os.path.isdir(base): continue
        for root,_,files in os.walk(base):
            for fn in files:
                yield os.path.join(root,fn)

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1<<20),b""): h.update(b)
    return h.hexdigest()

files=list(all_files())

if MODEL2_CKPT_OVERRIDE:
    PHASE1_CKPT=str(MODEL2_CKPT_OVERRIDE)
else:
    ckpts=[
        p for p in files if p.lower().endswith(".pt")
        and (
            "dec_full_best" in os.path.basename(p).lower()
            or ("model2" in p.lower() and "best" in os.path.basename(p).lower())
        )
    ]
    assert ckpts,"Attach Model2 dec_full_best*.pt or set MODEL2_CKPT_OVERRIDE."
    PHASE1_CKPT=sorted(
        ckpts,
        key=lambda p:(sum(w in p.lower() for w in ["model2","decoder-only","decoder_only","phase1"]),p),
        reverse=True,
    )[0]
    print("checkpoint candidates:")
    for p in ckpts: print("  -",p)

if TOKENIZER_OVERRIDE:
    TOKENIZER_FILE=str(TOKENIZER_OVERRIDE)
else:
    toks=[p for p in files if os.path.basename(p).lower()=="tokenizer.json"]
    assert toks,"Attach Model2 tokenizer.json or set TOKENIZER_OVERRIDE."
    cp_parts=set(Path(PHASE1_CKPT).parts)
    def tok_score(p):
        parts=set(Path(p).parts)
        return (len(cp_parts & parts),sum(w in p.lower() for w in ["model2","decoder","phase1"]),p)
    TOKENIZER_FILE=sorted(toks,key=tok_score,reverse=True)[0]
    print("tokenizer candidates:")
    for p in toks: print("  -",p)

assert os.path.isfile(PHASE1_CKPT)
assert os.path.isfile(TOKENIZER_FILE)
print("chosen checkpoint:",PHASE1_CKPT)
print("chosen tokenizer :",TOKENIZER_FILE)
print("tokenizer SHA256 :",sha256_file(TOKENIZER_FILE))
if not TOKENIZER_OVERRIDE:
    _sh={p:sha256_file(p) for p in toks}
    for p,h in _sh.items(): print("  tokenizer",h[:12],p)
    if len(set(_sh.values()))>1:
        print("⚠️ several DIFFERENT tokenizer.json files are attached (e.g. Model4 vs Model2). Make sure the chosen one is the Model2 tokenizer;",
              "a wrong tokenizer still loads the checkpoint strictly but gives garbage inputs. Cell 7 prints a Phase-1 LM-loss sanity check.")
if EXPECTED_TOKENIZER_SHA256:
    assert sha256_file(TOKENIZER_FILE)==EXPECTED_TOKENIZER_SHA256,"Tokenizer SHA mismatch"


checkpoint candidates:
  - /kaggle/input/datasets/aaryant23/phase2-model2-ckpts/dec_full_best (8).pt
chosen checkpoint: /kaggle/input/datasets/aaryant23/phase2-model2-ckpts/dec_full_best (8).pt
chosen tokenizer : /kaggle/input/datasets/aaryant23/phase2-model2-ckpts/tokenizer (1).json
tokenizer SHA256 : 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d


In [3]:
# CELL 3 — exact LegalBPE loader
BPE_SPECIALS=["<pad>","<unk>","<s>","</s>"]
class LegalBPE:
    def __init__(self,vocab_size=5000):
        self.vocab_size=int(vocab_size); self.merges=[]
        self.itos=list(BPE_SPECIALS); self.stoi={t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes={}
        for b in range(256):
            s=f"{b:02x}"; self.symbol_bytes[s]=bytes([b]); self.stoi[s]=len(self.itos); self.itos.append(s)
        self._reset_encode_cache()
    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)
    @staticmethod
    def word_to_symbols(word): return [f"{b:02x}" for b in word.encode("utf-8")]
    def _reset_encode_cache(self):
        self._ranks={(l,r):i for i,(l,r) in enumerate(self.merges)}
        self._word_cache={}; self._cache_nmerges=len(self.merges)
    def encode_word(self,word):
        if self._cache_nmerges!=len(self.merges): self._reset_encode_cache()
        hit=self._word_cache.get(word)
        if hit is not None: return list(hit)
        symbols=self.word_to_symbols(word)
        while len(symbols)>1:
            best=None; rank=None
            for j in range(len(symbols)-1):
                rr=self._ranks.get((symbols[j],symbols[j+1]))
                if rr is not None and (rank is None or rr<rank):
                    best=(symbols[j],symbols[j+1]); rank=rr
            if best is None: break
            l,r=best; merged=[]; j=0
            while j<len(symbols):
                if j+1<len(symbols) and symbols[j]==l and symbols[j+1]==r:
                    merged.append(l+r); j+=2
                else:
                    merged.append(symbols[j]); j+=1
            symbols=merged
        if len(self._word_cache)>=250000: self._word_cache.clear()
        self._word_cache[word]=tuple(symbols)
        return symbols
    def encode(self,text):
        out=[]; unk=self.unk_token_id
        for unit in re.findall(r"\S+|\s+",str(text)):
            for s in self.encode_word(unit): out.append(self.stoi.get(s,unk))
        return out
    def decode(self,ids):
        buf=bytearray()
        for idx in ids:
            tok=self.itos[int(idx)]
            if tok not in BPE_SPECIALS: buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8",errors="replace")
    @classmethod
    def load_json(cls,path):
        with open(path,encoding="utf-8") as f:
            obj=json.load(f)
        tok=cls(obj["vocab_size"])
        tok.merges=[list(x) for x in obj["merges"]]
        tok.itos=list(obj["itos"]); tok.stoi={t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size=len(tok.itos); tok.symbol_bytes={}
        for b in range(256):
            s=f"{b:02x}"; tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges: tok.symbol_bytes[l+r]=tok.symbol_bytes[l]+tok.symbol_bytes[r]
        tok._reset_encode_cache(); return tok

tokenizer=LegalBPE.load_json(TOKENIZER_FILE)
assert len(tokenizer)==VOCAB_SIZE
probe="IN THE SUPREME COURT OF INDIA\nSection 14: ₹100 — legal test."
assert tokenizer.decode(tokenizer.encode(probe))==probe
print("Tokenizer PASS:",len(tokenizer))


Tokenizer PASS: 24576


In [4]:
# CELL 4 — download Phase4 + build clean splits (+ train/test text normalisation)

FNS = {
    "train": "Phase_4_CJPE/Train/train.json",
    "dev":   "Phase_4_CJPE/Train/dev.json",
    "test":  "Phase_4_CJPE/Test/test.json",
}

paths = {
    k: hf_hub_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        filename=v,
        token=HF_TOKEN
    )
    for k, v in FNS.items()
}

train_raw = read_json(paths["train"])
dev_raw   = read_json(paths["dev"])
test_raw  = read_json(paths["test"])


def dedup_labeled(rows, name):
    seen = {}
    conflicts = set()
    text_diff = 0

    for r in rows:
        cid = str(r["id"])
        y   = int(r["label"])
        txt = str(r["text"])

        assert y in (0, 1)

        if cid in seen:
            if seen[cid]["label"] != y:
                conflicts.add(cid)

            elif seen[cid]["text"] != txt:
                text_diff += 1

                # Keep the more complete version if duplicate texts differ.
                if len(txt) > len(seen[cid]["text"]):
                    seen[cid]["text"] = txt
        else:
            seen[cid] = {
                "id": cid,
                "text": txt,
                "label": y,
            }

    # Drop IDs whose duplicate rows have conflicting labels.
    for cid in conflicts:
        seen.pop(cid, None)

    print(
        f"{name}: rows={len(rows)} "
        f"unique={len(seen)+len(conflicts)} | "
        f"same-ID different-text={text_diff} | "
        f"label-conflict IDs dropped={len(conflicts)}"
    )

    return list(seen.values())


# -----------------------------
# Deduplicate TRAIN + DEV
# -----------------------------
train_records = dedup_labeled(train_raw, "train")
dev_unique    = dedup_labeled(dev_raw, "dev")


# -----------------------------
# TEST
# Deliberately copy ONLY id/text.
# Never read/use TEST labels or expert fields.
# -----------------------------
test_records = [
    {
        "id": str(r["id"]),
        "text": str(r["text"]),
    }
    for r in test_raw
]

test_ids = {r["id"] for r in test_records}

assert len(test_ids) == len(test_records), (
    f"Duplicate TEST IDs found: rows={len(test_records)}, "
    f"unique={len(test_ids)}"
)


# -----------------------------
# Remove DEV ↔ TEST overlap
# -----------------------------
overlap_ids = {r["id"] for r in dev_unique} & test_ids
overlap_count = len(overlap_ids)

clean_dev = [
    r for r in dev_unique
    if r["id"] not in test_ids
]


# -----------------------------
# Dataset sanity checks
# -----------------------------
if len(train_records) != 32305:
    print(
        "⚠️ TRAIN unique count differs from earlier audit:",
        len(train_records)
    )

assert len(test_records) == 1517, (
    f"Unexpected TEST size: {len(test_records)}"
)

assert len(clean_dev) == 994, (
    f"Unexpected CLEAN DEV size: {len(clean_dev)}"
)

# Two known valid dataset layouts:
#
# 1) Older DEV:
#       dev_unique = 2511
#       overlap = 1517
#       clean_dev = 994
#
# 2) Newer/already-clean DEV:
#       dev_unique = 994
#       overlap = 0
#       clean_dev = 994
#
if overlap_count == 1517:
    print(
        "✅ Raw DEV contains 1517 TEST-overlap cases; "
        "they were excluded."
    )

elif overlap_count == 0 and len(dev_unique) == 994:
    print(
        "✅ DEV is already leakage-clean; "
        "no TEST-overlap cases are present."
    )

else:
    print(
        "⚠️ Unexpected DEV/TEST layout:",
        f"dev_unique={len(dev_unique)},",
        f"overlap={overlap_count},",
        f"clean_dev={len(clean_dev)}"
    )

    # Still require the final split itself to be leakage-safe.
    assert len(clean_dev) == 994


# Hard leakage guards
train_ids = {r["id"] for r in train_records}
clean_dev_ids = {r["id"] for r in clean_dev}

assert not (train_ids & test_ids), (
    f"TRAIN↔TEST overlap found: {len(train_ids & test_ids)} IDs"
)

assert not (clean_dev_ids & test_ids), (
    f"CLEAN DEV↔TEST overlap found: "
    f"{len(clean_dev_ids & test_ids)} IDs"
)


# -----------------------------
# Text-style diagnostics
# -----------------------------
def upper_ratio(t):
    t = t[:20000]
    letters = sum(c.isalpha() for c in t)
    up = sum(c.isupper() for c in t)
    return up / max(letters, 1)


_rng = random.Random(SEED)

def sample_ratio(rows, k=1500):
    idx = _rng.sample(
        range(len(rows)),
        min(k, len(rows))
    )
    return np.array([
        upper_ratio(rows[i]["text"])
        for i in idx
    ])


ur_tr = sample_ratio(train_records)
ur_dv = sample_ratio(clean_dev)
ur_te = sample_ratio(test_records)

print(
    f"uppercase-letter ratio  "
    f"train median={np.median(ur_tr):.4f} "
    f"(>2% in {100*np.mean(ur_tr > 0.02):.1f}% of docs)"
    f" | clean-dev median={np.median(ur_dv):.4f}"
    f" | test median={np.median(ur_te):.4f}"
)

print(
    "newline-containing docs  "
    "train: {:.1f}% | dev: {:.1f}% | test: {:.1f}%".format(
        *[
            100 * np.mean([
                "\n" in r["text"]
                for r in rows[:2000]
            ])
            for rows in (
                train_records,
                clean_dev,
                test_records
            )
        ]
    )
)


# -----------------------------
# Text normalisation
# -----------------------------
LOWERCASE = (
    CASE_MODE == "lower"
    or (
        CASE_MODE == "auto"
        and float(np.median(ur_te)) < 0.01
    )
)

print(
    "LOWERCASE:", LOWERCASE,
    "| COLLAPSE_WS:", COLLAPSE_WS
)


def normalize_text(t):
    if LOWERCASE:
        t = t.lower()

    if COLLAPSE_WS:
        t = re.sub(r"\s+", " ", t).strip()

    return t


for rows in (
    train_records,
    clean_dev,
    test_records
):
    for r in rows:
        r["text"] = normalize_text(r["text"])


# Free raw copies
del train_raw, dev_raw, test_raw, dev_unique
gc.collect()


# -----------------------------
# Final audit
# -----------------------------
print(
    "TRAIN:",
    len(train_records),
    Counter(r["label"] for r in train_records)
)

print(
    "CLEAN DEV:",
    len(clean_dev),
    Counter(r["label"] for r in clean_dev)
)

print(
    "TEST:",
    len(test_records)
)

print(
    "Excluded DEV↔TEST overlap:",
    overlap_count
)

print("✅ TEST label/expert fields ignored.")
print("✅ Final CLEAN DEV contains no TEST IDs.")

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

train: rows=37387 unique=32305 | same-ID different-text=0 | label-conflict IDs dropped=0
dev: rows=1988 unique=994 | same-ID different-text=0 | label-conflict IDs dropped=0
✅ DEV is already leakage-clean; no TEST-overlap cases are present.
uppercase-letter ratio  train median=0.0351 (>2% in 97.1% of docs) | clean-dev median=0.0000 | test median=0.0000
newline-containing docs  train: 0.0% | dev: 100.0% | test: 98.9%
LOWERCASE: True | COLLAPSE_WS: True
TRAIN: 32305 Counter({0: 18920, 1: 13385})
CLEAN DEV: 994 Counter({0: 497, 1: 497})
TEST: 1517
Excluded DEV↔TEST overlap: 0
✅ TEST label/expert fields ignored.
✅ Final CLEAN DEV contains no TEST IDs.


In [5]:
# CELL 5 — exact Phase-1 Model2 architecture + strict checkpoint load
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self,d_model,max_positions):
        super().__init__()
        pos=torch.arange(max_positions,dtype=torch.float32).unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2,dtype=torch.float32)*(-math.log(10000.0)/d_model))
        pe=torch.zeros(max_positions,d_model,dtype=torch.float32)
        pe[:,0::2]=torch.sin(pos*div)
        pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer("pe",pe,persistent=False)
    def forward(self,position_ids): return self.pe[position_ids]

class CausalSelfAttention(nn.Module):
    def __init__(self,d_model,n_heads,dropout):
        super().__init__()
        assert d_model%n_heads==0
        self.n_heads=n_heads; self.head_dim=d_model//n_heads
        self.qkv=nn.Linear(d_model,3*d_model)
        self.out_proj=nn.Linear(d_model,d_model)
        self.dropout=dropout
    def forward(self,x,past_kv=None,use_cache=False,key_valid=None):
        B,T,C=x.shape
        qkv=self.qkv(x).view(B,T,3,self.n_heads,self.head_dim)
        q,k,v=qkv.unbind(dim=2)
        q=q.transpose(1,2); k=k.transpose(1,2); v=v.transpose(1,2)
        past_len=0
        if past_kv is not None:
            pk,pv=past_kv; past_len=pk.size(2)
            k=torch.cat([pk,k],dim=2); v=torch.cat([pv,v],dim=2)
        attn_mask=None; is_causal=False
        if key_valid is not None:
            key_mask=key_valid[:,None,None,:]
            if past_len==0 and T>1:
                causal=torch.ones((T,T),dtype=torch.bool,device=x.device).tril()
                attn_mask=key_mask & causal[None,None,:,:]
                eye=torch.eye(T,dtype=torch.bool,device=x.device)[None,None,:,:]
                attn_mask=attn_mask|eye
            else: attn_mask=key_mask
        elif past_kv is None:
            is_causal=True
        y=F.scaled_dot_product_attention(
            q,k,v,attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )
        y=y.transpose(1,2).contiguous().view(B,T,C)
        y=self.out_proj(y)
        return y,((k,v) if use_cache else None)

class DecoderBlock(nn.Module):
    def __init__(self,d_model,n_heads,ffn_dim,dropout):
        super().__init__()
        self.ln1=nn.LayerNorm(d_model)
        self.attn=CausalSelfAttention(d_model,n_heads,dropout)
        self.ln2=nn.LayerNorm(d_model)
        self.ff=nn.Sequential(
            nn.Linear(d_model,ffn_dim),nn.GELU(),
            nn.Linear(ffn_dim,d_model),nn.Dropout(dropout),
        )
        self.resid_drop=nn.Dropout(dropout)
    def forward(self,x,past_kv=None,use_cache=False,key_valid=None):
        a,new_kv=self.attn(self.ln1(x),past_kv=past_kv,use_cache=use_cache,key_valid=key_valid)
        x=x+self.resid_drop(a)
        x=x+self.ff(self.ln2(x))
        return x,new_kv

class DecoderOnlyLM(nn.Module):
    def __init__(self,vocab_size,d_model,n_layers,n_heads,ffn_dim,dropout,max_positions):
        super().__init__()
        self.vocab_size=vocab_size; self.d_model=d_model; self.max_positions=max_positions
        self.token_emb=nn.Embedding(vocab_size,d_model)
        self.pos=SinusoidalPositionalEncoding(d_model,max_positions)
        self.drop=nn.Dropout(dropout)
        self.blocks=nn.ModuleList([DecoderBlock(d_model,n_heads,ffn_dim,dropout) for _ in range(n_layers)])
        self.final_ln=nn.LayerNorm(d_model)
        self.apply(self._init_weights)
        scaled_std=0.02/math.sqrt(2*n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight,mean=0.0,std=scaled_std)
            nn.init.normal_(block.ff[2].weight,mean=0.0,std=scaled_std)
    def _init_weights(self,m):
        if isinstance(m,nn.Linear):
            nn.init.normal_(m.weight,mean=0.0,std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m,nn.Embedding):
            nn.init.normal_(m.weight,mean=0.0,std=0.02)
    def forward_hidden(self,input_ids,past_key_values=None,use_cache=False,position_ids=None,key_valid=None):
        B,T=input_ids.shape
        past_len=0 if past_key_values is None else past_key_values[0][0].size(2)
        if past_len+T>self.max_positions:
            raise ValueError(f"context {past_len+T} exceeds max_positions={self.max_positions}")
        if position_ids is None:
            position_ids=torch.arange(past_len,past_len+T,device=input_ids.device)[None,:].expand(B,T)
        x=self.token_emb(input_ids)+self.pos(position_ids).to(self.token_emb.weight.dtype)
        x=self.drop(x)
        new_cache=[] if use_cache else None
        for i,block in enumerate(self.blocks):
            past=None if past_key_values is None else past_key_values[i]
            x,kv=block(x,past_kv=past,use_cache=use_cache,key_valid=key_valid)
            if use_cache:new_cache.append(kv)
        x=self.final_ln(x)
        return x,new_cache
    def project(self,hidden): return F.linear(hidden,self.token_emb.weight)

def build_backbone():
    return DecoderOnlyLM(
        vocab_size=VOCAB_SIZE,d_model=D_MODEL,n_layers=N_LAYERS,n_heads=N_HEADS,
        ffn_dim=FFN_DIM,dropout=DROPOUT,max_positions=MAX_POSITIONS,
    )

obj=torch.load(PHASE1_CKPT,map_location="cpu",weights_only=False)
state=obj["model"] if isinstance(obj,dict) and "model" in obj else obj
assert tuple(state["token_emb.weight"].shape)==(VOCAB_SIZE,D_MODEL)
assert len(tokenizer)==VOCAB_SIZE
bb=build_backbone()
bb.load_state_dict(state,strict=True)
n_params=sum(p.numel() for p in bb.parameters())
assert 75_000_000<=n_params<=85_000_000,n_params
print(f"Strict Model2 load PASS | params={n_params/1e6:.3f}M")
if isinstance(obj,dict) and "config" in obj: print("checkpoint config:",obj["config"])
del obj,state; gc.collect()


Strict Model2 load PASS | params=79.736M
checkpoint config: {'architecture': 'decoder_only_causal', 'vocab_size': 24576, 'd_model': 640, 'n_layers': 13, 'n_heads': 10, 'ffn_dim': 2560, 'dropout': 0.0, 'max_positions': 2048, 'seq_len': 1536, 'prompt_p10_tokens': 1050, 'prompt_p95_tokens': 1139, 'expected_cont_tokens': 384, 'prompt_split_min': 1050, 'prompt_split_max': 1139}


0

In [6]:
# CELL 6 — AdaLoRA + adaptive global rank allocator + hierarchical classifier
class AdaLoRALinear(nn.Module):
    def __init__(self,base,r=ADALORA_INIT_R,alpha=ADALORA_ALPHA,dropout=ADALORA_DROPOUT):
        super().__init__()
        assert isinstance(base,nn.Linear)
        self.base=base
        for p in self.base.parameters(): p.requires_grad=False
        self.r=int(r); self.alpha=float(alpha); self.scale=self.alpha/self.r
        self.adalora_dropout=nn.Dropout(float(dropout))
        self.lora_A=nn.Parameter(torch.empty(self.r,base.in_features))
        self.lora_B=nn.Parameter(torch.zeros(base.out_features,self.r))
        self.lora_E=nn.Parameter(torch.ones(self.r))
        self.register_buffer("rank_mask",torch.ones(self.r,dtype=torch.float32))
        self.register_buffer("importance_ema",torch.zeros(self.r,dtype=torch.float32))
        nn.init.kaiming_uniform_(self.lora_A,a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)
    def forward(self,x):
        z=self.base(x)
        a=F.linear(self.adalora_dropout(x),self.lora_A)
        coeff=(self.lora_E*self.rank_mask).to(a.dtype)
        delta=F.linear(a*coeff,self.lora_B)
        return z+self.scale*delta
    @torch.no_grad()
    def observe_importance(self,beta=IMPORTANCE_BETA):
        if self.lora_E.grad is None:return
        s=(self.lora_E.detach()*self.lora_E.grad.detach()).abs().float()
        if not bool(torch.isfinite(s).all()): return   # AMP-overflow step: keep EMA clean
        self.importance_ema.mul_(beta).add_(s,alpha=1-beta)
    @property
    def active_rank(self): return int(self.rank_mask.sum().item())

def inject_adalora(backbone):
    for p in backbone.parameters():p.requires_grad=False
    modules=[]
    for bi,b in enumerate(backbone.blocks):
        specs=[
            (b.attn,"qkv",f"blocks.{bi}.attn.qkv"),
            (b.attn,"out_proj",f"blocks.{bi}.attn.out_proj"),
            (b.ff,0,f"blocks.{bi}.ff.0"),
            (b.ff,2,f"blocks.{bi}.ff.2"),
        ]
        for parent,key,path in specs:
            base=parent[key] if isinstance(key,int) else getattr(parent,key)
            wrapped=AdaLoRALinear(base)
            if isinstance(key,int):parent[key]=wrapped
            else:setattr(parent,key,wrapped)
            modules.append((path,wrapped))
    print("AdaLoRA modules:",len(modules))
    return backbone,modules

class AdaptiveRankAllocator:
    def __init__(self,modules,total_updates):
        self.modules=list(modules); self.total_updates=int(total_updates)
        self.initial=sum(m.r for _,m in self.modules)
        self.target=len(self.modules)*ADALORA_TARGET_R
        self.warm=max(1,int(self.total_updates*RANK_WARMUP_FRAC))
        self.final=max(1,int(self.total_updates*RANK_FINAL_FRAC))
    def budget(self,step):
        if step<=self.warm:return self.initial
        end=max(self.warm+1,self.total_updates-self.final)
        if step>=end:return self.target
        p=(step-self.warm)/max(1,end-self.warm)
        return int(round(self.target+(self.initial-self.target)*(1-p)**3))
    @torch.no_grad()
    def prune(self,step,force=False):
        if not force and step%RANK_UPDATE_INTERVAL!=0:
            return self.summary()
        budget=max(self.target,min(self.initial,self.budget(step)))
        scores=[]
        for mi,(_,m) in enumerate(self.modules):
            s=m.importance_ema.detach().cpu()+1e-8*m.lora_E.detach().abs().cpu()
            for ri,v in enumerate(s.tolist()):scores.append((float(v),mi,ri))
        masks=[torch.zeros_like(m.rank_mask) for _,m in self.modules]
        chosen=set()
        for mi,(_,m) in enumerate(self.modules):
            s=m.importance_ema.detach().cpu()+1e-8*m.lora_E.detach().abs().cpu()
            ri=int(torch.argmax(s).item())
            masks[mi][ri]=1; chosen.add((mi,ri))
        rem=max(0,budget-len(self.modules))
        for _,mi,ri in sorted(scores,key=lambda x:(x[0],-x[1],-x[2]),reverse=True):
            if rem<=0:break
            if (mi,ri) in chosen:continue
            masks[mi][ri]=1; chosen.add((mi,ri)); rem-=1
        for mask,(_,m) in zip(masks,self.modules):
            m.rank_mask.copy_(mask.to(m.rank_mask.device))
        summary=self.summary()
        assert summary["min"]>=1,summary
        assert summary["active_total"]==budget,(summary,budget)
        return summary
    def summary(self):
        r=[m.active_rank for _,m in self.modules]
        return {"active_total":int(sum(r)),"modules":len(r),"min":int(min(r)),"mean":float(np.mean(r)),"max":int(max(r))}

class HierarchicalCJPE(nn.Module):
    def __init__(self,backbone):
        super().__init__()
        self.backbone=backbone
        self.chunk_proj=nn.Sequential(
            nn.Linear(2*D_MODEL,CHUNK_PROJ_DIM),nn.GELU(),
            nn.LayerNorm(CHUNK_PROJ_DIM),nn.Dropout(HEAD_DROPOUT),
        )
        self.chunk_pos=nn.Embedding(N_CHUNKS,CHUNK_PROJ_DIM)
        self.attn_score=nn.Sequential(nn.Linear(CHUNK_PROJ_DIM,128),nn.Tanh(),nn.Linear(128,1))
        self.fc=nn.Sequential(
            nn.Linear(2*CHUNK_PROJ_DIM,256),nn.GELU(),
            nn.Dropout(HEAD_DROPOUT),nn.Linear(256,2),
        )
    def forward(self,input_ids,valid):
        B,K,T=input_ids.shape
        x=input_ids.view(B*K,T); m=valid.view(B*K,T)
        h,_=self.backbone.forward_hidden(x,key_valid=None)
        mf=m.unsqueeze(-1).to(h.dtype)
        mean=(h*mf).sum(1)/mf.sum(1).clamp(min=1)
        last_idx=m.long().sum(1).clamp(min=1)-1
        last=h[torch.arange(B*K,device=h.device),last_idx]
        z=self.chunk_proj(torch.cat([last.float(),mean.float()],-1)).view(B,K,-1)
        z=z+self.chunk_pos(torch.arange(K,device=z.device)[None,:])
        w=torch.softmax(self.attn_score(z).squeeze(-1),-1)
        att=(z*w.unsqueeze(-1)).sum(1); mx=z.max(1).values
        return self.fc(torch.cat([att,mx],-1)),w

bb,adalora_modules=inject_adalora(bb)
model=HierarchicalCJPE(bb).to(DEVICE)

adapter_params=sum(p.numel() for n,p in model.named_parameters() if p.requires_grad and n.startswith("backbone."))
head_params=sum(p.numel() for n,p in model.named_parameters() if p.requires_grad and not n.startswith("backbone."))
total=sum(p.numel() for p in model.parameters())
print(f"AdaLoRA trainable={adapter_params/1e6:.3f}M | head={head_params/1e6:.3f}M | total model={total/1e6:.3f}M")


AdaLoRA modules: 52
AdaLoRA trainable=1.598M | head=0.741M | total model=82.075M


In [7]:
# CELL 7 — tokenize/select HEAD-MIDDLE-EVIDENCE-TAIL (slice-based: no need to BPE-encode 400k-word cases)
SLICE_CHARS=CHUNK_LEN*8     # ~3k chars >> 384 BPE tokens

def _snap_forward(text,i):
    i=max(0,min(int(i),max(0,len(text)-1)))
    m=re.search(r"\s",text[i:i+400])
    return i+m.end() if m else i

def _enc(t):
    return tokenizer.encode(t)

def _window_start(text,center,width):
    s=max(0,int(center)-width//2)
    return _snap_forward(text,s)

def _last_evidence_position(text):
    low=text.lower()
    best=-1
    for cue in EVIDENCE_CUES:
        p=low.rfind(cue)
        if p>best:
            best=p
    return best

SELECT_STATS=Counter()   # diagnostics only

def select_chunks(text):
    """
    Four views of a long case:
      head / middle / evidence-or-late / tail.
    For short cases we derive all views from exact full BPE tokens.
    """
    n=len(text); W=SLICE_CHARS

    if n<=4*W:
        SELECT_STATS["short_doc"]+=1
        ids=_enc(text) or [tokenizer.unk_token_id]
        L=len(ids)

        if L<=CHUNK_LEN:
            ch=[ids,ids,ids,ids]
        else:
            def tok_window(frac):
                center=int(round(frac*(L-1)))
                s=max(0,min(L-CHUNK_LEN,center-CHUNK_LEN//2))
                return ids[s:s+CHUNK_LEN]

            # evidence view falls back to a late (75%) view for short docs.
            ch=[
                ids[:CHUNK_LEN],
                tok_window(0.50),
                tok_window(0.75),
                ids[-CHUNK_LEN:],
            ]
        est=L

    else:
        # Prefix
        head=_enc(text[:W])

        # True character-centred middle slice.
        ms=_window_start(text,n*0.50,W)
        mid=_enc(text[ms:ms+W])

        # Evidence chunk: centre around the LAST neutral disposition cue.
        # If absent, use a late-document (75%) region.
        ep=_last_evidence_position(text)
        SELECT_STATS["long_doc"]+=1; SELECT_STATS["cue_found" if ep>=0 else "cue_missing"]+=1
        ec=(ep if ep>=0 else int(n*0.75))
        es=_window_start(text,ec,W)
        evidence=_enc(text[es:es+W])

        # Suffix
        ts=_snap_forward(text,max(0,n-W))
        tail=_enc(text[ts:])

        if min(len(head),len(mid),len(evidence),len(tail))<CHUNK_LEN:
            # Rare token-density/pathological fallback: exact full tokenisation.
            SELECT_STATS["fallback_full_tokenisation"]+=1
            ids=_enc(text) or [tokenizer.unk_token_id]
            L=len(ids)
            def tok_window(frac):
                center=int(round(frac*(L-1)))
                s=max(0,min(max(0,L-CHUNK_LEN),center-CHUNK_LEN//2))
                return ids[s:s+CHUNK_LEN]
            ch=[
                ids[:CHUNK_LEN],
                tok_window(0.50),
                tok_window(0.75),
                ids[-CHUNK_LEN:],
            ]
            est=L
        else:
            ch=[
                head[:CHUNK_LEN],
                mid[:CHUNK_LEN],
                evidence[:CHUNK_LEN],
                tail[-CHUNK_LEN:],
            ]
            est=int(n*len(head)/max(W,1))  # statistics only

    return [np.asarray(x,dtype=np.uint16) for x in ch],est

class CaseDataset(Dataset):
    def __init__(self,records,labeled):
        self.records=records; self.labeled=labeled; self.chunks=[]; self.full_lens=[]
        SELECT_STATS.clear(); t0=time.time()
        for i,r in enumerate(records):
            ch,L=select_chunks(r["text"]); self.chunks.append(ch); self.full_lens.append(L)
            if (i+1)%5000==0: print(f"tokenized {i+1}/{len(records)} elapsed={time.time()-t0:.0f}s",flush=True)
        print(f"done {len(records)} in {time.time()-t0:.0f}s | est. median BPE={np.median(self.full_lens):.0f} p95={np.percentile(self.full_lens,95):.0f}")
        print("   chunk-selection stats:",dict(SELECT_STATS))
    def __len__(self): return len(self.records)
    def __getitem__(self,i):
        r=self.records[i]; d={"id":r["id"],"chunks":self.chunks[i]}
        if self.labeled: d["label"]=int(r["label"])
        return d

def collate_cases(batch):
    B=len(batch)
    ids=torch.full((B,N_CHUNKS,CHUNK_LEN),tokenizer.pad_token_id,dtype=torch.long)
    valid=torch.zeros((B,N_CHUNKS,CHUNK_LEN),dtype=torch.bool)
    for b,x in enumerate(batch):
        for k,c in enumerate(x["chunks"]):
            z=torch.from_numpy(c.astype(np.int64)); n=min(len(z),CHUNK_LEN)
            ids[b,k,:n]=z[:n]; valid[b,k,:n]=True
    out={"input_ids":ids,"valid":valid,"ids":[x["id"] for x in batch]}
    if "label" in batch[0]:
        out["labels"]=torch.tensor([x["label"] for x in batch],dtype=torch.long)
    return out

train_ds=CaseDataset(train_records,True)
dev_ds=CaseDataset(clean_dev,True)
test_ds=CaseDataset(test_records,False)

g=torch.Generator(); g.manual_seed(SEED)
_kw=dict(collate_fn=collate_cases,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=="cuda",persistent_workers=NUM_WORKERS>0)
train_loader=DataLoader(train_ds,batch_size=MICRO_BATCH_SIZE,shuffle=True,generator=g,**_kw)
dev_loader=DataLoader(dev_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,**_kw)
test_loader=DataLoader(test_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,**_kw)

print("batches:",len(train_loader),len(dev_loader),len(test_loader))

# Verify the v2 fast causal-kernel optimisation on a real padded batch.
# Valid tokens in a causal model cannot attend to future right-padding.
@torch.no_grad()
def verify_fast_causal_equivalence():
    b=next(iter(dev_loader))
    x=b["input_ids"][:1].to(DEVICE)
    v=b["valid"][:1].to(DEVICE)
    B,K,T=x.shape
    xf=x.view(B*K,T); vf=v.view(B*K,T)
    model.backbone.eval()
    with torch.amp.autocast("cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
        h_mask,_=model.backbone.forward_hidden(xf,key_valid=vf)
        h_fast,_=model.backbone.forward_hidden(xf,key_valid=None)
    d=(h_mask[vf]-h_fast[vf]).abs().max().item()
    print("fast causal-kernel valid-token max |Δ|:",d)
    assert d < 5e-2,(
        "Fast causal path deviates unexpectedly; restore key_valid masking."
    )
    model.train()

verify_fast_causal_equivalence()

# Sanity: the frozen Phase-1 LM should predict next tokens far better than chance (ln V = 10.1).
# At this point every AdaLoRA B is zero, so this is exactly the Phase-1 model.
@torch.no_grad()
def phase1_lm_sanity(n_docs=16):
    model.backbone.eval(); losses=[]
    for i in range(min(n_docs,len(train_ds))):
        ids=torch.from_numpy(train_ds.chunks[i][1].astype(np.int64))[None].to(DEVICE)   # middle view
        if ids.size(1)<32: continue
        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
            h,_=model.backbone.forward_hidden(ids)
            lg=model.backbone.project(h[:,:-1]).float()
        losses.append(F.cross_entropy(lg[0],ids[0,1:]).item())
    model.train()
    if not losses: print("LM sanity skipped (docs too short)"); return
    m=float(np.mean(losses))
    print(f"Phase-1 LM sanity: next-token loss={m:.3f} (ppl {math.exp(m):.1f}) | chance={math.log(VOCAB_SIZE):.2f}")
    if m>7.0:
        print("⚠️ LM loss is unusually high; tokenizer/text normalization may not match Phase-1 Model2.")
    assert m<8.5, (
        f"Phase-1 LM sanity FAILED: mean next-token loss={m:.3f}. "
        "Most likely the wrong tokenizer.json was selected. "
        "Set TOKENIZER_OVERRIDE or EXPECTED_TOKENIZER_SHA256 to the Model2 tokenizer."
    )
phase1_lm_sanity()


tokenized 5000/32305 elapsed=14s
tokenized 10000/32305 elapsed=25s
tokenized 15000/32305 elapsed=37s
tokenized 20000/32305 elapsed=51s
tokenized 25000/32305 elapsed=63s
tokenized 30000/32305 elapsed=74s
done 32305 in 78s | est. median BPE=4416 p95=20119
   chunk-selection stats: {'long_doc': 15940, 'cue_found': 12469, 'cue_missing': 3471, 'short_doc': 16365}
done 994 in 3s | est. median BPE=6232 p95=21137
   chunk-selection stats: {'long_doc': 657, 'cue_found': 480, 'cue_missing': 177, 'short_doc': 337}
done 1517 in 4s | est. median BPE=6505 p95=22861
   chunk-selection stats: {'long_doc': 1053, 'cue_found': 772, 'short_doc': 464, 'cue_missing': 281}
batches: 16153 166 253
fast causal-kernel valid-token max |Δ|: 0.0
Phase-1 LM sanity: next-token loss=2.656 (ppl 14.2) | chance=10.11


In [8]:
# CELL 8 — optimizer, exact metric, threshold search
counts=Counter(r["label"] for r in train_records)
class_w=torch.tensor([len(train_records)/(2*counts[i]) for i in [0,1]],dtype=torch.float32,device=DEVICE)
class_w/=class_w.mean()
print("class weights:",class_w.tolist())

def make_groups(m):
    ada=[p for n,p in m.named_parameters() if p.requires_grad and n.startswith("backbone.")]
    head=[p for n,p in m.named_parameters() if p.requires_grad and not n.startswith("backbone.")]
    return [
        {"params":ada,"lr":ADALORA_LR,"weight_decay":0.0,"name":"adalora"},
        {"params":head,"lr":HEAD_LR,"weight_decay":WEIGHT_DECAY,"name":"head"},
    ]

groups=make_groups(model)
try: optimizer=torch.optim.AdamW(groups,fused=(DEVICE.type=="cuda"))
except Exception: optimizer=torch.optim.AdamW(groups)

updates_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)
total_updates=NUM_EPOCHS*updates_per_epoch
warmup=max(1,int(total_updates*WARMUP_RATIO))
rank_allocator=AdaptiveRankAllocator(adalora_modules,total_updates)

def lr_mult(step):
    if step<warmup:return max(1e-8,step/warmup)
    p=(step-warmup)/max(1,total_updates-warmup)
    return 0.5*(1+math.cos(math.pi*min(max(p,0),1)))

scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_mult)
amp_enabled=USE_AMP and DEVICE.type=="cuda"
scaler=torch.amp.GradScaler("cuda",enabled=amp_enabled,init_scale=AMP_INIT_SCALE,growth_interval=2000)

def macro_f1(y,p):return f1_score(y,p,average="macro",labels=[0,1],zero_division=0)
def search_threshold(y,prob,smooth=THRESH_SMOOTH):
    fs=np.array([macro_f1(y,(prob>=t).astype(np.int64)) for t in THRESHOLDS])
    k=max(1,int(smooth))
    if k>1:
        if k%2==0:k+=1
        pad=k//2
        sm=np.convolve(np.pad(fs,(pad,pad),mode="edge"),np.ones(k)/k,mode="valid")
    else:sm=fs
    i=int(np.argmax(sm)); t=float(THRESHOLDS[i])
    return float(fs[i]),t,(prob>=t).astype(np.int64),float(fs.max())

@torch.no_grad()
def predict_logits(m,loader):
    m.eval(); lgs=[]; ys=[]; ids=[]; ats=[]
    for b in loader:
        x=b["input_ids"].to(DEVICE,non_blocking=True); v=b["valid"].to(DEVICE,non_blocking=True)
        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=amp_enabled):
            lg,a=m(x,v)
        lgs.append(lg.float().cpu().numpy()); ats.append(a.float().cpu().numpy()); ids.extend(b["ids"])
        if "labels" in b:ys.append(b["labels"].numpy())
    return np.concatenate(lgs),(np.concatenate(ys) if ys else None),ids,np.concatenate(ats)

def _p1(lg):
    lg=lg.astype(np.float64); lg=lg-lg.max(-1,keepdims=True); e=np.exp(lg)
    return (e[:,1]/e.sum(-1)).astype(np.float32)
def _logit(p):
    p=np.clip(np.asarray(p,dtype=np.float64),1e-6,1-1e-6)
    return np.log(p/(1-p))
def predict_probs(m,loader):
    lg,y,ids,att=predict_logits(m,loader)
    return _p1(lg),y,ids,att

print("updates/epoch:",updates_per_epoch,"total updates:",total_updates)
print("rank budget:",rank_allocator.initial,"→",rank_allocator.target)


class weights: [0.8286643028259277, 1.1713356971740723]
updates/epoch: 2020 total updates: 4040
rank budget: 624 → 208


In [9]:
# CELL 9 — train / adaptive rank / DEV selection
BEST=ROOT/"best_adalora_cjpe.pt"
BEST_TMP=BEST.with_suffix(".pt.tmp")
if BEST_TMP.exists():BEST_TMP.unlink()

def trainable_state_dict(m):
    keep=("lora_A","lora_B","lora_E","rank_mask","importance_ema")
    return {k:v.detach().cpu() for k,v in m.state_dict().items()
            if any(x in k for x in keep) or k.startswith(("chunk_proj.","chunk_pos.","attn_score.","fc."))}

def load_trainable_state(m,state):
    missing,unexpected=m.load_state_dict(state,strict=False)
    assert not unexpected,unexpected
    req=[k for k in m.state_dict()
         if any(x in k for x in ("lora_A","lora_B","lora_E","rank_mask","importance_ema"))
         or k.startswith(("chunk_proj.","chunk_pos.","attn_score.","fc."))]
    bad=[k for k in req if k in missing]
    assert not bad,bad[:10]

history=[]; best_f1=-1.0; best_thr=.5; best_frac=None
bad=0; global_update=0; overflows=0; nonfinite_skipped=0
optimizer.zero_grad(set_to_none=True)

eval_steps=[]
for j in range(1,EVAL_PER_EPOCH+1):
    s=int(round(len(train_loader)*j/EVAL_PER_EPOCH))
    if s<len(train_loader):s=max(GRAD_ACCUM_STEPS,(s//GRAD_ACCUM_STEPS)*GRAD_ACCUM_STEPS)
    else:s=len(train_loader)
    eval_steps.append(s)
eval_steps=sorted(set(eval_steps))
print("DEV eval batches:",eval_steps)

stop=False
for epoch in range(1,NUM_EPOCHS+1):
    model.train(); run=0.; nr=0; ep0=time.time(); good=0
    for step,b in enumerate(train_loader,1):
        x=b["input_ids"].to(DEVICE,non_blocking=True); v=b["valid"].to(DEVICE,non_blocking=True)
        y=b["labels"].to(DEVICE,non_blocking=True)
        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=amp_enabled):
            lg,_=model(x,v)
            raw=F.cross_entropy(lg.float(),y,weight=class_w,label_smoothing=LABEL_SMOOTHING)
            loss=raw/GRAD_ACCUM_STEPS
        raw_val=float(raw.detach())
        if math.isfinite(raw_val):
            scaler.scale(loss).backward(); good+=1; run+=raw_val; nr+=1
        else:
            nonfinite_skipped+=1
            print(f"⚠️ non-finite loss epoch={epoch} batch={step}; micro-batch skipped (total {nonfinite_skipped})")

        # NOTE: no `continue` above — the accumulation boundary, logging and DEV eval below must still run.
        if step%GRAD_ACCUM_STEPS==0 or step==len(train_loader):
            if good==0:
                optimizer.zero_grad(set_to_none=True)
            else:
                scaler.unscale_(optimizer)
                for _,am in adalora_modules:am.observe_importance()
                gn=torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],GRAD_CLIP)
                finite=bool(torch.isfinite(torch.as_tensor(gn)).item())
                old=float(scaler.get_scale()); scaler.step(optimizer); scaler.update(); new=float(scaler.get_scale())
                optimizer.zero_grad(set_to_none=True)
                if finite and new>=old:
                    scheduler.step(); global_update+=1; rank_allocator.prune(global_update)
                else:overflows+=1
            good=0

        if step%500==0 or step==len(train_loader):
            elapsed=time.time()-ep0; eta=(elapsed/max(step,1))*(len(train_loader)-step)
            print(f"epoch={epoch} batch={step}/{len(train_loader)} loss={run/max(nr,1):.4f} "
                  f"elapsed={elapsed/60:.1f}m ETA_epoch={eta/60:.1f}m "
                  f"rank_mean={rank_allocator.summary()['mean']:.2f}",flush=True)

        if step in eval_steps:
            prob,ydev,idsdev,att=predict_probs(model,dev_loader)
            f,t,pred,grid_best=search_threshold(ydev,prob)
            row={
                "epoch":epoch,"epoch_frac":(epoch-1)+step/len(train_loader),"batch":step,
                "macro_f1":f,"threshold":t,"grid_best_macro_f1":grid_best,
                "f1_at_0.5":float(macro_f1(ydev,(prob>=.5).astype(np.int64))),
                "train_loss":run/max(nr,1),"global_update":global_update,
                "rank":rank_allocator.summary(),"amp_overflows":overflows,
                "nonfinite_skipped":nonfinite_skipped,"attention_mean":att.mean(0).tolist(),
            }
            history.append(row)
            save_json(history,ROOT/"history.json",indent=2)
            print("DEV:",row)
            if f>best_f1+1e-12:
                best_f1=f; best_thr=t; best_frac=row["epoch_frac"]; bad=0
                torch.save({
                    "trainable_state":trainable_state_dict(model),
                    "best_macro_f1":float(best_f1),"best_threshold":float(best_thr),
                    "best_epoch_frac":float(best_frac),"rank_summary":rank_allocator.summary(),
                    "adalora_init_r":ADALORA_INIT_R,"adalora_target_r":ADALORA_TARGET_R,
                    "adalora_alpha":ADALORA_ALPHA,"adalora_targets":list(ADALORA_TARGETS),
                    "seed":SEED,"method":"AdaLoRA adaptive-rank",
                },BEST_TMP)
                os.replace(BEST_TMP,BEST)
                print(f"✅ NEW BEST Macro-F1={best_f1:.6f} threshold={best_thr:.3f}")
            else:bad+=1
            if bad>=PATIENCE_EVALS:
                print("Early stopping."); stop=True; break
            model.train()
    if stop:break

assert BEST.is_file()
print("DONE best F1/thr/frac:",best_f1,best_thr,best_frac)


DEV eval batches: [5384, 10768, 16153]
epoch=1 batch=500/16153 loss=0.7101 elapsed=1.0m ETA_epoch=30.6m rank_mean=12.00
epoch=1 batch=1000/16153 loss=0.7117 elapsed=2.0m ETA_epoch=30.5m rank_mean=12.00
epoch=1 batch=1500/16153 loss=0.7139 elapsed=3.0m ETA_epoch=29.7m rank_mean=12.00
epoch=1 batch=2000/16153 loss=0.7123 elapsed=4.1m ETA_epoch=28.9m rank_mean=12.00
epoch=1 batch=2500/16153 loss=0.7116 elapsed=5.1m ETA_epoch=27.9m rank_mean=12.00
epoch=1 batch=3000/16153 loss=0.7095 elapsed=6.2m ETA_epoch=27.0m rank_mean=12.00
epoch=1 batch=3500/16153 loss=0.7071 elapsed=7.2m ETA_epoch=26.0m rank_mean=11.79
epoch=1 batch=4000/16153 loss=0.7032 elapsed=8.2m ETA_epoch=25.0m rank_mean=11.21
epoch=1 batch=4500/16153 loss=0.6999 elapsed=9.3m ETA_epoch=24.0m rank_mean=10.75
epoch=1 batch=5000/16153 loss=0.6961 elapsed=10.3m ETA_epoch=23.0m rank_mean=10.31
DEV: {'epoch': 1, 'epoch_frac': 0.333312697331765, 'batch': 5384, 'macro_f1': 0.5449840493612133, 'threshold': 0.32, 'grid_best_macro_f1': 0.

In [10]:
# CELL 10 — exact supplied Phase-4 evaluator on clean DEV
best_obj=torch.load(BEST,map_location="cpu",weights_only=False)
load_trainable_state(model,best_obj["trainable_state"])
best_thr=float(best_obj["best_threshold"])

dev_prob,dev_y,dev_ids,dev_att=predict_probs(model,dev_loader)
dev_pred=(dev_prob>=best_thr).astype(np.int64)

truth=[{"id":str(i),"gold":int(y)} for i,y in zip(dev_ids,dev_y)]
pred=[{"id":str(i),"pred":int(p)} for i,p in zip(dev_ids,dev_pred)]

EVALUATOR_CODE=r'''#!/usr/bin/env python3
import argparse,json,sys
from sklearn.metrics import f1_score,precision_score,recall_score
def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or isinstance(r['gold'],bool) or r['gold'] not in (0,1): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        out[r['id']]=r['gold']
    return out
def read_pred(path):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or isinstance(r['pred'],bool) or r['pred'] not in (0,1): raise ValueError(f'Invalid prediction item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
        out[r['id']]=r['pred']
    return out
def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True)
    a=ap.parse_args()
    t=read_truth(a.truth); p=read_pred(a.prediction)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=[t[i] for i in t]; yp=[p[i] for i in t]
    print(f'Examples: {len(t)}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",labels=[0,1],zero_division=0):.6f}')
    print(f'Precision: {precision_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
    print(f'Recall: {recall_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e:
        print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)
'''

with tempfile.TemporaryDirectory() as td:
    tp=Path(td)/"truth.json"
    pp=Path(td)/"prediction.json"
    ep=Path(td)/"evaluate_phase4.py"
    save_json(truth,tp)
    save_json(pred,pp)
    ep.write_text(EVALUATOR_CODE,encoding="utf-8")
    res=subprocess.run(
        [sys.executable,str(ep),"--truth",str(tp),"--prediction",str(pp)],
        capture_output=True,text=True,
    )
    print(res.stdout)
    if res.stderr: print(res.stderr)
    assert res.returncode==0

print("✅ Exact Phase-4 evaluator PASS")
np.savez_compressed(
    ROOT/"dev_probs.npz",
    prob=dev_prob.astype(np.float32),
    y=dev_y.astype(np.int64),
    id=np.asarray(dev_ids,dtype=str),   # loads without allow_pickle
    threshold=np.asarray([best_thr],dtype=np.float32),
    chunk_attention=dev_att.astype(np.float32),
)


Examples: 994
Macro-F1: 0.652899
Precision: 0.672686
Recall: 0.599598

✅ Exact Phase-4 evaluator PASS


In [11]:
# CELL 11 — fresh Model2+AdaLoRA and clean TRAIN+DEV refit
def fresh_model():
    b=build_backbone()
    o=torch.load(PHASE1_CKPT,map_location="cpu",weights_only=False)
    st=o["model"] if isinstance(o,dict) and "model" in o else o
    b.load_state_dict(st,strict=True)
    b,mods=inject_adalora(b)
    return HierarchicalCJPE(b).to(DEVICE),mods

refit_model=None; refit_modules=None
REFIT_SEED=SEED+1

if RUN_REFIT:
    all_records=train_records+clean_dev
    all_ds=torch.utils.data.ConcatDataset([train_ds,dev_ds])
    seed_everything(REFIT_SEED)
    gg=torch.Generator();gg.manual_seed(REFIT_SEED)
    all_loader=DataLoader(
        all_ds,batch_size=MICRO_BATCH_SIZE,shuffle=True,generator=gg,
        collate_fn=collate_cases,num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type=="cuda",persistent_workers=NUM_WORKERS>0,
    )
    refit_model,refit_modules=fresh_model()
    try:ropt=torch.optim.AdamW(make_groups(refit_model),fused=(DEVICE.type=="cuda"))
    except Exception:ropt=torch.optim.AdamW(make_groups(refit_model))

    target_batches=max(
        GRAD_ACCUM_STEPS,
        int(round(float(best_obj["best_epoch_frac"])*len(all_loader)/GRAD_ACCUM_STEPS))*GRAD_ACCUM_STEPS,
    )
    upe_all=math.ceil(len(all_loader)/GRAD_ACCUM_STEPS)
    horizon_updates=(NUM_EPOCHS*upe_all) if REFIT_MATCH_ORIGINAL_SCHEDULE else math.ceil(target_batches/GRAD_ACCUM_STEPS)
    rw=max(1,int(horizon_updates*WARMUP_RATIO))
    def rmult(step):
        if step<rw:return max(1e-8,step/rw)
        p=(step-rw)/max(1,horizon_updates-rw)
        return .5*(1+math.cos(math.pi*min(max(p,0),1)))
    rsched=torch.optim.lr_scheduler.LambdaLR(ropt,rmult)
    rscaler=torch.amp.GradScaler("cuda",enabled=amp_enabled,init_scale=AMP_INIT_SCALE,growth_interval=2000)
    ralloc=AdaptiveRankAllocator(refit_modules,horizon_updates)

    ac=Counter(r["label"] for r in all_records)
    cw=torch.tensor([len(all_records)/(2*ac[i]) for i in [0,1]],dtype=torch.float32,device=DEVICE)
    cw/=cw.mean()

    print("refit target micro-batches:",target_batches,"| LR/rank horizon updates:",horizon_updates)
    ropt.zero_grad(set_to_none=True); seen=0; rupd=0; over=0; nonfin=0; good=0; t0=time.time()

    while seen<target_batches:
        refit_model.train()
        for b in all_loader:
            if seen>=target_batches:break
            x=b["input_ids"].to(DEVICE,non_blocking=True); v=b["valid"].to(DEVICE,non_blocking=True)
            y=b["labels"].to(DEVICE,non_blocking=True)
            with torch.amp.autocast("cuda",dtype=torch.float16,enabled=amp_enabled):
                lg,_=refit_model(x,v)
                raw=F.cross_entropy(lg.float(),y,weight=cw,label_smoothing=LABEL_SMOOTHING)
                loss=raw/GRAD_ACCUM_STEPS
            seen+=1
            if math.isfinite(float(raw.detach())):
                rscaler.scale(loss).backward(); good+=1
            else:
                nonfin+=1

            if seen%GRAD_ACCUM_STEPS==0 or seen>=target_batches:
                if good==0:
                    ropt.zero_grad(set_to_none=True)
                else:
                    rscaler.unscale_(ropt)
                    for _,am in refit_modules:am.observe_importance()
                    gn=torch.nn.utils.clip_grad_norm_([p for p in refit_model.parameters() if p.requires_grad],GRAD_CLIP)
                    finite=bool(torch.isfinite(torch.as_tensor(gn)).item())
                    old=float(rscaler.get_scale());rscaler.step(ropt);rscaler.update();new=float(rscaler.get_scale())
                    ropt.zero_grad(set_to_none=True)
                    if finite and new>=old:
                        rsched.step();rupd+=1;ralloc.prune(rupd)
                    else:over+=1
                good=0

            if seen%500==0 or seen>=target_batches:
                rate=(time.time()-t0)/max(seen,1)
                print(f"refit {seen}/{target_batches} ETA={(target_batches-seen)*rate/60:.1f}m "
                      f"rank_mean={ralloc.summary()['mean']:.2f} overflow={over}",flush=True)

    FINAL=ROOT/"final_refit_adalora_cjpe.pt";TMP=FINAL.with_suffix(".pt.tmp")
    if TMP.exists():TMP.unlink()
    torch.save({
        "trainable_state":trainable_state_dict(refit_model),
        "source_best_epoch_frac":float(best_obj["best_epoch_frac"]),
        "rank_summary":ralloc.summary(),
        "adalora_init_r":ADALORA_INIT_R,
        "adalora_target_r":ADALORA_TARGET_R,
        "adalora_alpha":ADALORA_ALPHA,
        "adalora_targets":list(ADALORA_TARGETS),
        "refit_seed":REFIT_SEED,
        "refit_target_batches":int(target_batches),
        "overflows":over,"nonfinite_skipped":nonfin,
        "method":"AdaLoRA adaptive-rank",
    },TMP)
    os.replace(TMP,FINAL)
    print("✅ refit complete | final rank:",ralloc.summary())


AdaLoRA modules: 52
refit target micro-batches: 11096 | LR/rank horizon updates: 4164
refit 500/11096 ETA=21.9m rank_mean=12.00 overflow=0
refit 1000/11096 ETA=20.9m rank_mean=12.00 overflow=0
refit 1500/11096 ETA=19.8m rank_mean=12.00 overflow=0
refit 2000/11096 ETA=18.8m rank_mean=12.00 overflow=0
refit 2500/11096 ETA=17.8m rank_mean=12.00 overflow=0
refit 3000/11096 ETA=16.7m rank_mean=12.00 overflow=0
refit 3500/11096 ETA=15.7m rank_mean=11.88 overflow=0
refit 4000/11096 ETA=14.7m rank_mean=11.33 overflow=0
refit 4500/11096 ETA=13.6m rank_mean=10.87 overflow=0
refit 5000/11096 ETA=12.6m rank_mean=10.44 overflow=0
refit 5500/11096 ETA=11.6m rank_mean=10.02 overflow=0
refit 6000/11096 ETA=10.5m rank_mean=9.56 overflow=0
refit 6500/11096 ETA=9.5m rank_mean=9.17 overflow=0
refit 7000/11096 ETA=8.5m rank_mean=8.81 overflow=0
refit 7500/11096 ETA=7.4m rank_mean=8.48 overflow=0
refit 8000/11096 ETA=6.4m rank_mean=8.10 overflow=0
refit 8500/11096 ETA=5.4m rank_mean=7.79 overflow=0
refit 90

In [12]:
# CELL 12 — TEST inference + label-free calibration
held_lg,_,held_ids,held_att=predict_logits(model,test_loader)
assert held_ids==[r["id"] for r in test_records] and len(held_ids)==1517
held_prob=_p1(held_lg)

refit_prob=None; refit_thr=None; test_refit_agreement=None
if refit_model is not None:
    refit_lg,_,rid,_=predict_logits(refit_model,test_loader)
    assert rid==held_ids
    refit_prob=_p1(refit_lg)
    held_rate=float((held_prob>=best_thr).mean())
    refit_thr=float(np.clip(np.quantile(refit_prob,1-held_rate),.05,.95))
    test_refit_agreement=float(((held_prob>=best_thr)==(refit_prob>=refit_thr)).mean())
    print(f"heldout positive rate={held_rate:.3f} | refit thr={refit_thr:.3f} | agreement={test_refit_agreement:.4f}")

np.savez_compressed(
    ROOT/"test_probs.npz",heldout=held_prob.astype(np.float32),
    refit=(refit_prob.astype(np.float32) if refit_prob is not None else np.array([],dtype=np.float32)),
    id=np.asarray(held_ids,dtype=str),
    heldout_threshold=np.asarray([best_thr],dtype=np.float32),
    refit_threshold=np.asarray([refit_thr if refit_thr is not None else np.nan],dtype=np.float32),
)


heldout positive rate=0.454 | refit thr=0.422 | agreement=0.7943


In [13]:
# CELL 13 — three high-value Codabench ZIPs
official_ids=[r["id"] for r in test_records]
assert len(official_ids)==len(set(official_ids))==1517

def write_zip(filename,labels):
    labels=np.asarray(labels,dtype=np.int64)
    rec=[{"id":str(cid),"pred":int(y)} for cid,y in zip(official_ids,labels)]
    assert len(rec)==1517 and len({r["id"] for r in rec})==1517
    assert all(type(r["pred"]) is int and r["pred"] in (0,1) for r in rec)
    zp=ROOT/filename
    with tempfile.TemporaryDirectory() as td:
        jp=Path(td)/"prediction.json"
        save_json(rec,jp)
        with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z:
            z.write(jp,arcname="prediction.json")
    with zipfile.ZipFile(zp,"r") as z:
        assert z.namelist()==["prediction.json"],z.namelist()
        inside=json.loads(z.read("prediction.json").decode("utf-8"))
        assert len(inside)==1517
        assert {r["id"] for r in inside}==set(official_ids)
    print(f"✅ {filename} | positive={labels.mean():.4f} | {Counter(labels.tolist())}")
    return labels

if refit_prob is not None:
    margin=.5*(_logit(held_prob)-_logit(best_thr))+.5*(_logit(refit_prob)-_logit(refit_thr))
    y_best=write_zip("submission_best.zip",(margin>=0).astype(np.int64))
else:
    y_best=write_zip("submission_best.zip",(held_prob>=best_thr).astype(np.int64))

y_held=write_zip("submission_heldout.zip",(held_prob>=best_thr).astype(np.int64))

if refit_prob is not None:
    margin_h70_r30=.70*(_logit(held_prob)-_logit(best_thr)) + .30*(_logit(refit_prob)-_logit(refit_thr))
    y_hedge=write_zip("submission_margin_h70_r30.zip",(margin_h70_r30>=0).astype(np.int64))
else:
    y_hedge=None

disagree_best_held=float(np.mean(y_best!=y_held))
disagree_best_hedge=(None if y_hedge is None else float(np.mean(y_best!=y_hedge)))
disagree_held_hedge=(None if y_hedge is None else float(np.mean(y_held!=y_hedge)))
print("best vs heldout disagreement:",disagree_best_held)
if y_hedge is not None:
    print("best vs 70/30 hedge disagreement:",disagree_best_hedge)
    print("heldout vs 70/30 hedge disagreement:",disagree_held_hedge)


✅ submission_best.zip | positive=0.4463 | Counter({0: 840, 1: 677})
✅ submission_heldout.zip | positive=0.4542 | Counter({0: 828, 1: 689})
✅ submission_margin_h70_r30.zip | positive=0.4436 | Counter({0: 844, 1: 673})
best vs heldout disagreement: 0.05009887936717205
best vs 70/30 hedge disagreement: 0.02109426499670402
heldout vs 70/30 hedge disagreement: 0.02900461437046803


In [14]:
# CELL 14 — manifest
manifest={
    "phase":4,"experiment":3,"method":"AdaLoRA adaptive-rank low-rank adaptation",
    "phase1_model":"Model2 decoder-only causal Transformer",
    "checkpoint":str(PHASE1_CKPT),"tokenizer_sha256":sha256_file(TOKENIZER_FILE),
    "train_unique":len(train_records),"clean_dev":len(clean_dev),"test":len(test_records),
    "excluded_dev_test_overlap":overlap_count,"chunks":CHUNK_NAMES,"chunk_len":CHUNK_LEN,
    "adalora_init_r":ADALORA_INIT_R,"adalora_target_r":ADALORA_TARGET_R,
    "adalora_alpha":ADALORA_ALPHA,"adalora_dropout":ADALORA_DROPOUT,
    "adalora_targets":list(ADALORA_TARGETS),
    "best_dev_macro_f1":float(best_obj["best_macro_f1"]),
    "best_threshold":float(best_obj["best_threshold"]),
    "best_epoch_frac":float(best_obj["best_epoch_frac"]),
    "best_rank_summary":best_obj.get("rank_summary"),
    "lowercase":bool(LOWERCASE),"collapse_ws":bool(COLLAPSE_WS),
    "refit_run":bool(refit_model is not None),
    "refit_threshold":(None if refit_thr is None else float(refit_thr)),
    "test_refit_agreement":test_refit_agreement,
    "refit_target_batches":(int(target_batches) if refit_model is not None else None),
    "test_positive_rate_best":float(np.mean(y_best)),
    "submission_strategies":{
        "submission_best.zip":"50/50 calibrated heldout+refit margin ensemble",
        "submission_heldout.zip":"clean-DEV-selected heldout checkpoint",
        "submission_margin_h70_r30.zip":"70/30 calibrated heldout/refit margin hedge",
    },
    "submission_disagreement":{
        "best_vs_heldout":disagree_best_held,
        "best_vs_h70_r30":disagree_best_hedge,
        "heldout_vs_h70_r30":disagree_held_hedge,
    },
    "torch_version":torch.__version__,
    "seed":SEED,"test_labels_used":False,"overlap_dev_labels_used":False,
}
save_json(manifest,ROOT/"manifest.json",indent=2)
print(json.dumps(manifest,indent=2))

print("\nESSENTIAL OUTPUTS:")
for fn in [
    "best_adalora_cjpe.pt","final_refit_adalora_cjpe.pt",
    "dev_probs.npz","test_probs.npz","history.json","manifest.json",
    "submission_best.zip","submission_heldout.zip","submission_margin_h70_r30.zip",
]:
    print(("✅" if (ROOT/fn).exists() else "—"),fn)


{
  "phase": 4,
  "experiment": 3,
  "method": "AdaLoRA adaptive-rank low-rank adaptation",
  "phase1_model": "Model2 decoder-only causal Transformer",
  "checkpoint": "/kaggle/input/datasets/aaryant23/phase2-model2-ckpts/dec_full_best (8).pt",
  "tokenizer_sha256": "9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d",
  "train_unique": 32305,
  "clean_dev": 994,
  "test": 1517,
  "excluded_dev_test_overlap": 0,
  "chunks": [
    "head",
    "middle",
    "evidence",
    "tail"
  ],
  "chunk_len": 384,
  "adalora_init_r": 12,
  "adalora_target_r": 4,
  "adalora_alpha": 24.0,
  "adalora_dropout": 0.05,
  "adalora_targets": [
    "attn.qkv",
    "attn.out_proj",
    "ff.0",
    "ff.2"
  ],
  "best_dev_macro_f1": 0.6528991391911645,
  "best_threshold": 0.385,
  "best_epoch_frac": 0.66662539466353,
  "best_rank_summary": {
    "active_total": 333,
    "modules": 52,
    "min": 1,
    "mean": 6.403846153846154,
    "max": 12
  },
  "lowercase": true,
  "collapse_ws": true,
  "

## Keep / submit

Submit in this order:

1. **`submission_best.zip`** — primary 50/50 calibrated heldout+refit margin ensemble.
2. **`submission_margin_h70_r30.zip`** — conservative hedge that trusts the clean-DEV-selected model more.
3. **`submission_heldout.zip`** — pure clean-DEV-selected heldout model.

Keep the two AdaLoRA/head checkpoints, `dev_probs.npz`, `test_probs.npz`,
`history.json`, `manifest.json`, and whichever submission ZIPs you score.
The original Phase-1 Model2 checkpoint + tokenizer are still required to reconstruct the frozen backbone.
